In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config('spark.shuffle.useOldFetchProtocol','true'). \
config("spark.sql.warehouse.dir", f"/user/itv026931/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [43]:
spark.sql("""select member_id, count(*) as num_customers from itv026931_lending_club.customers
group by member_id having num_customers > 1 order by num_customers desc""")

member_id,num_customers
afcc02d37c15a4737...,5
9eab3a65c61d16216...,4
fa41e1ac4acd20940...,4
b45af2c038ff03f24...,4
7848c5e6bb69fbc96...,3
19df0e31f4fb2617a...,3
8a105c87d497ba399...,3
27ed1904bb3cf0f3d...,3
f136f8f17fbf6ca67...,3
c53ecc2494a825fe6...,3


In [6]:
spark.sql("select * from  itv026931_lending_club.customers where member_id like 'afcc02d37c15a4737%'")

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,total_high_credit_limit,application_type,join_annual_income,verification_status_joint,ingest_date
afcc02d37c15a4737...,null,6,MORTGAGE,55000.0,IL,604xx,USA,B,B5,Verified,207300.0,Individual,null,null,2026-09-27 13:39:...
afcc02d37c15a4737...,null,6,MORTGAGE,55000.0,IL,604xx,USA,B,B5,Verified,110907.0,Individual,null,null,2026-09-27 13:39:...
afcc02d37c15a4737...,null,6,MORTGAGE,55000.0,IL,604xx,USA,B,B5,Verified,138780.0,Individual,null,null,2026-09-27 13:39:...
afcc02d37c15a4737...,null,6,MORTGAGE,55000.0,IL,604xx,USA,B,B5,Verified,129833.0,Individual,null,null,2026-09-27 13:39:...
afcc02d37c15a4737...,null,6,MORTGAGE,55000.0,IL,604xx,USA,B,B5,Verified,171165.0,Individual,null,null,2026-09-27 13:39:...


In [9]:
spark.sql("select * from itv026931_lending_club.loans_defaulters_delinq where member_id like 'afcc02d37c15a4737%'")

member_id,delinq_2yrs,delinq_amnt,mths_since_last_delinq
afcc02d37c15a4737...,1,0.0,37
afcc02d37c15a4737...,3,0.0,13
afcc02d37c15a4737...,1,0.0,16


In [10]:
spark.sql("""select member_id, count(*) as num_customers from itv026931_lending_club.loans_defaulters_detail_rec_enq_nar
group by member_id having num_customers > 1 order by num_customers desc""")

member_id,num_customers
e3b0c44298fc1c149...,33
afcc02d37c15a4737...,5
fa41e1ac4acd20940...,4
9eab3a65c61d16216...,4
b45af2c038ff03f24...,4
19df0e31f4fb2617a...,3
27ed1904bb3cf0f3d...,3
d3c23d16a29d76646...,3
ccd1c1e56db83e2b8...,3
f136f8f17fbf6ca67...,3


In [11]:
spark.sql("""select * from itv026931_lending_club.loans_defaulters_detail_rec_enq_nar where member_id like 'afcc02d37c15a4737%'""")

member_id,pub_rec,pub_rec_bankruptcies,inq_last_6mths
afcc02d37c15a4737...,0,0,0
afcc02d37c15a4737...,0,0,1
afcc02d37c15a4737...,0,0,0
afcc02d37c15a4737...,0,0,3
afcc02d37c15a4737...,0,0,2


In [2]:
bad_data_customer_df = spark.sql("""select member_id from (select member_id, count(*) as total from itv026931_lending_club.customers
group by member_id having total > 1)""")

In [15]:
bad_data_customer_df.count()

3157

In [3]:
bad_data_loans_defaulters_delinq_df = spark.sql("""select member_id from (select member_id, count(*) as total from itv026931_lending_club.loans_defaulters_delinq
group by member_id having total > 1)""")

In [23]:
bad_data_loans_defaulters_delinq_df.count()

939

In [4]:
bad_data_loans_defaulters_detail_rec_enq_nar_df = spark.sql("""select member_id from (select member_id, count(*) as total from itv026931_lending_club.loans_defaulters_detail_rec_enq_nar
group by member_id having total > 1)""")

In [25]:
bad_data_loans_defaulters_detail_rec_enq_nar_df.count()

3189

In [29]:
bad_data_customer_df.repartition(1).write \
.format("csv") \
.option("header",True) \
.mode("overwrite") \
.option("path","/user/itv026931/lendingclubproject/bad/bad_data_customers") \
.save()

In [32]:
bad_data_loans_defaulters_delinq_df.repartition(1).write \
.format("csv") \
.mode("overwrite") \
.option("header",True) \
.option("path","/user/itv026931/lendingclubproject/bad/bad_data_loans_defaulters_delinq") \
.save()

In [39]:
bad_data_loans_defaulters_detail_rec_enq_nar_df.repartition(1).write \
.format("csv") \
.mode("overwrite") \
.option("header",True) \
.option("path","/user/itv026931/lendingclubproject/bad/bad_data_loans_defaulters_detail_rec_enq_nar") \
.save()

In [5]:
bad_customers_data_df = bad_data_customer_df.select("member_id") \
.union(bad_data_loans_defaulters_delinq_df.select("member_id")) \
.union(bad_data_loans_defaulters_detail_rec_enq_nar_df.select("member_id"))

In [6]:
bad_customers_data_final_df = bad_customers_data_df.distinct()

In [37]:
bad_customers_data_final_df.count()

3189

In [7]:
bad_customers_data_final_df.repartition(1).write \
.format("csv") \
.option("header",True) \
.mode("overwrite") \
.option("path","/user/itv026931/lendingclubproject/bad/bad_customers_data_final_df") \
.save()

 CLEANING GOOD DATA FROM BAD TABLE

In [8]:
bad_customers_data_final_df.createOrReplaceTempView("bad_data_customer")

In [12]:
customers_df = spark.sql("select * from itv026931_lending_club.customers where member_id NOT IN(select member_id from bad_data_customer)")

In [13]:
customers_df

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,total_high_credit_limit,application_type,join_annual_income,verification_status_joint,ingest_date
26e9702b9953b38fc...,Associate Director,10,MORTGAGE,260000.0,CA,907xx,USA,B,B1,Verified,725965.0,Individual,null,null,2026-09-27 13:39:...
01f9345544e7119fa...,Legal Assistant,1,MORTGAGE,65000.0,TN,380xx,USA,A,A4,Not Verified,265434.0,Individual,null,null,2026-09-27 13:39:...
edab22b138beeb2dc...,Human Resourses,1,MORTGAGE,110000.0,OR,973xx,USA,D,D3,Not Verified,362023.0,Individual,null,null,2026-09-27 13:39:...
19290bbd251c1cca9...,Commercial Pilot,7,RENT,92000.0,MT,599xx,USA,D,D4,Source Verified,83462.0,Individual,null,null,2026-09-27 13:39:...
fd6758fcf19d3a517...,Supervisor,5,MORTGAGE,67762.0,TX,786xx,USA,C,C4,Verified,278125.0,Individual,null,null,2026-09-27 13:39:...
18bbb2002f7fe591b...,CT Technician,1,MORTGAGE,62000.0,TX,751xx,USA,D,D3,Verified,102395.0,Individual,null,null,2026-09-27 13:39:...
3bf7a045144c64d3e...,Manager,3,RENT,100000.0,CA,926xx,USA,D,D2,Not Verified,100296.0,Individual,null,null,2026-09-27 13:39:...
f380edbd0407baaf0...,Contractor,10,RENT,40000.0,GA,315xx,USA,B,B2,Source Verified,10120.0,Individual,null,null,2026-09-27 13:39:...
d555f060129f1b4b4...,Body shop robot o...,5,MORTGAGE,49000.0,TN,373xx,USA,E,E2,Verified,114240.0,Individual,null,null,2026-09-27 13:39:...
3f9ed30e58d1f0ae4...,owner,10,MORTGAGE,50000.0,PA,190xx,USA,A,A3,Not Verified,293000.0,Individual,null,null,2026-09-27 13:39:...


In [52]:
customers_df.write \
.mode("overwrite") \
.format("parquet") \
.option("path","/user/itv026931/lendingclubproject/cleaned_new/customer_parquet") \
.save()

In [10]:
loan_defaulters_delinq_df = spark.sql("""select * from itv026931_lending_club.loans_defaulters_delinq where member_id NOT IN
(select member_id from bad_data_customer)""")

In [54]:
loan_defaulters_delinq_df.write \
.mode("overwrite") \
.format("parquet") \
.option("path","/user/itv026931/lendingclubproject/cleaned_new/loan_defaulters_delinq_parquet") \
.save()

In [11]:
loan_defaulters_delinq_detail_rec_enq_df = spark.sql("""select * from itv026931_lending_club.loans_defaulters_detail_rec_enq_nar where member_id NOT IN
(select member_id from bad_data_customer)""")

In [56]:
loan_defaulters_delinq_detail_rec_enq_df.write \
.mode("overwrite") \
.format("parquet") \
.option("path","/user/itv026931/lendingclubproject/cleaned_new/loan_defaulters_detail_rec_enq_parquet") \
.save()

In [14]:
spark.sql("""
Create External Table itv026931_lending_club.customers_new(member_id string, emp_title string, emp_length int, home_ownership string, 
annual_income float, address_state string, address_zipcode string, address_country string, grade string, 
sub_grade string, verification_status string, total_high_credit_limit float, application_type string, 
join_annual_income float, verification_status_joint string, ingest_date timestamp)
STORED AS parquet
LOCATION '/user/itv026931/lendingclubproject/cleaned_new/customer_parquet'
""")

""


In [15]:
spark.sql("""
Create External Table itv026931_lending_club.loans_defaulters_delinq_new
(member_id string,delinq_2yrs integer, delinq_amnt float, mths_since_last_delinq integer)
STORED AS parquet
LOCATION '/user/itv026931/lendingclubproject/cleaned_new/loan_defaulters_delinq_parquet'
""")

""


In [16]:
spark.sql("""
Create External Table itv026931_lending_club.loans_defaulters_detail_rec_enq_nar_new
(member_id string, pub_rec integer, pub_rec_bankruptcies integer, inq_last_6mths integer)
STORED AS parquet
LOCATION '/user/itv026931/lendingclubproject/cleaned_new/loan_defaulters_detail_rec_enq_parquet'
""")

""


In [17]:
spark.sql("Select member_id, count(*) as count from itv026931_lending_club.customers_new group by member_id having count > 1")

member_id,count
